# Hybrid Recommender System with LLMs and Weaviate

This notebook demonstrates a hybrid recommendation system that combines:
- **Weaviate vector database** for hybrid search (BM25 + semantic vectors)
- **OpenAI LLM** for intelligent product suggestions
- **Amazon Product Sales Dataset** as the product catalog

**Pipeline:** User query → LLM generates product suggestions → Weaviate hybrid search retrieves real catalog matches.

## 1. Setup and Dependencies

In [ ]:
%pip install -r requirements.txt -q

In [ ]:
import os
import ast

import weaviate
import weaviate.classes.config as wc
import weaviate.classes.query as wq
from weaviate.util import generate_uuid5

import openai
from openai import OpenAI

import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv
from IPython.display import HTML

print(f"Weaviate client version: {weaviate.__version__}")
print(f"OpenAI version: {openai.__version__}")

## 2. Configuration

Load API keys from a `.env` file. Required variables:
- `OPENAI_API_KEY`
- `WCS_URL` — Weaviate Cloud Services cluster URL
- `WCS_API_KEY` — Weaviate Cloud Services API key

In [ ]:
load_dotenv()

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
WEAVIATE_URL = os.environ["WCS_URL"]
WEAVIATE_API_KEY = os.environ["WCS_API_KEY"]

chat_client = OpenAI()

## 3. Connect to Weaviate

Connect to a Weaviate Cloud Services (WCS) cluster. You can create a free sandbox at [console.weaviate.cloud](https://console.weaviate.cloud).

In [ ]:
weaviate_client = weaviate.connect_to_wcs(
    cluster_url=WEAVIATE_URL,
    auth_credentials=weaviate.auth.AuthApiKey(WEAVIATE_API_KEY),
    headers={
        "Content-Type": "application/json",
        "X-OpenAI-Api-Key": OPENAI_API_KEY,
    },
)

assert weaviate_client.is_live(), "Weaviate cluster is not reachable"
print("Connected to Weaviate cluster.")

## 4. Load and Clean Dataset

The [Amazon Product Sales Dataset](https://www.kaggle.com/) contains ~550K products across 142 categories with product names, categories, images, links, ratings, and prices.

In [ ]:
dataset = pd.read_csv(os.path.join("dataset", "Amazon-Products.csv"))
print(f"Raw dataset: {dataset.shape[0]:,} rows")
dataset.head()

In [ ]:
print(f"Duplicated rows: {dataset.duplicated().sum()}")
print(f"Missing values per column:\n{dataset.isna().sum()}")

In [ ]:
INR_TO_USD = 84


def inr_to_usd(amount_str: str) -> float:
    """Convert an INR price string like '₹32,999' to a USD float."""
    amount_inr = float(amount_str.replace("\u20b9", "").replace(",", ""))
    return round(amount_inr / INR_TO_USD, 2)


# Drop rows with missing values and work on a clean copy
df = dataset.dropna().copy()

# Convert prices from INR to USD
df["actual_price"] = df["actual_price"].apply(inr_to_usd)
df["discount_price"] = df["discount_price"].apply(inr_to_usd)

# Fix numeric types
df["ratings"] = pd.to_numeric(df["ratings"], errors="coerce")
df["no_of_ratings"] = df["no_of_ratings"].str.replace(",", "").astype(float)

# Drop the unnamed index column
df = df.drop(columns=["Unnamed: 0"], errors="ignore")

print(f"Cleaned dataset: {df.shape[0]:,} rows")
df.dtypes

## 5. Create Weaviate Collection

Define the `Products` collection schema. Text fields (`name`, `main_category`, `sub_category`) are vectorized using OpenAI's `text2vec-openai`; metadata fields are stored but excluded from vectorization.

In [ ]:
COLLECTION_NAME = "Products"

properties = [
    wc.Property(name="name", data_type=wc.DataType.TEXT),
    wc.Property(name="main_category", data_type=wc.DataType.TEXT),
    wc.Property(name="sub_category", data_type=wc.DataType.TEXT),
    wc.Property(name="image", data_type=wc.DataType.TEXT, skip_vectorization=True),
    wc.Property(name="link", data_type=wc.DataType.TEXT, skip_vectorization=True),
    wc.Property(name="ratings", data_type=wc.DataType.NUMBER, skip_vectorization=True),
    wc.Property(name="no_of_ratings", data_type=wc.DataType.NUMBER, skip_vectorization=True),
    wc.Property(name="discount_price", data_type=wc.DataType.NUMBER, skip_vectorization=True),
    wc.Property(name="actual_price", data_type=wc.DataType.NUMBER, skip_vectorization=True),
]

try:
    weaviate_client.collections.create(
        name=COLLECTION_NAME,
        properties=properties,
        vectorizer_config=wc.Configure.Vectorizer.text2vec_openai(),
    )
    print(f"{COLLECTION_NAME} collection created.")
except Exception as e:
    print(f"Collection already exists or creation failed: {e}")

## 6. Insert Products into Weaviate

To keep costs and time reasonable, we filter to the *grocery & gourmet foods* category and sample 1,000 products.

In [ ]:
CATEGORY = "grocery & gourmet foods"
SAMPLE_SIZE = 1000

df_sample = df[df["main_category"] == CATEGORY].sample(n=SAMPLE_SIZE, random_state=42)
print(f"Sampled {len(df_sample)} products from '{CATEGORY}'")

In [ ]:
products = weaviate_client.collections.get(COLLECTION_NAME)

with products.batch.dynamic() as batch:
    for _, row in tqdm(df_sample.iterrows(), total=len(df_sample)):
        batch.add_object(
            properties={
                "name": row["name"],
                "main_category": row["main_category"],
                "sub_category": row["sub_category"],
                "image": row["image"],
                "link": row["link"],
                "ratings": row["ratings"],
                "no_of_ratings": row["no_of_ratings"],
                "discount_price": row["discount_price"],
                "actual_price": row["actual_price"],
            },
            uuid=generate_uuid5(row["link"]),
        )

failed = len(products.batch.failed_objects)
print(f"Inserted {SAMPLE_SIZE - failed} products ({failed} failed).")

## 7. Hybrid Search

Weaviate's hybrid search combines **BM25** (keyword matching) with **vector search** (semantic similarity) to deliver more accurate and comprehensive results than either method alone.

In [ ]:
def display_results(response_objects):
    """Render search results as an HTML table with product images."""
    rows = []
    for item in response_objects:
        p = item.properties
        score = f"{item.metadata.score:.3f}" if item.metadata and item.metadata.score is not None else "N/A"
        rows.append({
            "Name": p["name"],
            "Image": f'<img src="{p["image"]}" width="100"/>',
            "Score": score,
            "Actual Price ($)": p["actual_price"],
            "Discount Price ($)": p["discount_price"],
        })
    return HTML(pd.DataFrame(rows).to_html(escape=False, index=False))

In [ ]:
QUERY = "chicken noodles"

products = weaviate_client.collections.get(COLLECTION_NAME)
response = products.query.hybrid(
    query=QUERY,
    limit=5,
    return_metadata=wq.MetadataQuery(score=True),
)

print(f"Hybrid search results for: '{QUERY}'")
display_results(response.objects)

## 8. LLM-Enhanced Recommendations

Use an LLM to generate product suggestions from the query, then search for those suggestions in the Weaviate catalog to find real matching products.

In [ ]:
SYSTEM_PROMPT = (
    "You are an expert in recommending grocery and gourmet food products. "
    "Based on the following query, provide a list of high-quality grocery "
    "and gourmet food items that would be ideal for the customer. "
    'Return JSON only with key "rec_prod" and a list of product names as values.'
)


def get_llm_suggestions(query: str) -> list[str]:
    """Ask the LLM for product name suggestions based on a query."""
    completion = chat_client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Query: {query}"},
        ],
        response_format={"type": "json_object"},
    )
    content = completion.choices[0].message.content
    return ast.literal_eval(content)["rec_prod"]

In [ ]:
suggestions = get_llm_suggestions(QUERY)
print(f"LLM suggestions for '{QUERY}':")
for i, name in enumerate(suggestions, 1):
    print(f"  {i}. {name}")

In [ ]:
# Use the last LLM suggestion to find real products via hybrid search
best_suggestion = suggestions[-1]
print(f"Searching catalog for LLM suggestion: '{best_suggestion}'\n")

response = products.query.hybrid(
    query=best_suggestion,
    limit=5,
    return_metadata=wq.MetadataQuery(score=True),
)

display_results(response.objects)

## 9. Near-Object Similarity Search

Use Weaviate's `near_object` to find products similar to the top result. This expands recommendations beyond keyword overlap by leveraging vector proximity in the embedding space.

In [ ]:
top_result_uuid = response.objects[0].uuid
print(f"Finding products similar to: '{response.objects[0].properties['name']}'\n")

similar_response = products.query.near_object(
    near_object=top_result_uuid,
    limit=5,
    return_metadata=wq.MetadataQuery(score=True),
)

display_results(similar_response.objects)

## 10. Cleanup

In [ ]:
weaviate_client.close()
print("Weaviate connection closed.")